In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

laoder = PyPDFLoader(PDF_PATH)
pages = laoder.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars)---")
print(pages[1].page_content[:500])

C:\Users\ANKUR\AppData\Local\Temp\ipykernel_14520\1494142285.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from the PDF.

--- First page preview (first 500 chars)---
Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "],
)

chunks = splitter.split_documents(pages)
len(chunks)

37

In [7]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vector_store= Chroma.from_documents(chunks,embeddings)

print(f"Vector store ready . {vector_store._collection.count()} vector stored")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15347.92it/s]


Vector store ready . 37 vector stored


In [9]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3}) #k=3 : top 3 searchs

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call c

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [11]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to partner networks abroad. Here’s how it works and the charges involved:

### **How It Works**  
1. **Authentication**: The visited network verifies your subscription via signaling protocols (SS7/Diameter), and your home network authorizes service.  
2. **Traffic Handling**: All data, voice, and SMS are tunneled back to your home network for billing, which may add latency compared to local networks.  

### **Charges**  
- **Roaming Zones**:  
  - **Zone A** (EU, UK, Australia, New Zealand): Lowest rates.  
  - **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
  - **Zone C** (Rest of the world): Highest per-MB/minute charges.  
- **Bundles**: Purchase a roaming bundle before traveling to Zone B/C to avoid high standard rates. Pre-bundle usage is charged at standard rates and not reversed.  

### **Common Issues**  
- Roaming disabled on your accou

In [12]:
debug_question = "What security measures protect against SIM swap fraud?"

docs = retriever.invoke(debug_question)
print(f"Question: {debug_question}")
print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"{'='*60}")
    print(f"Chunk {i} (page {doc.metadata.get('page', '?')})")
    print(f"{'='*60}")
    print(doc.page_content)
    print()

print("\nFinal Answer:")
print(chain.invoke(debug_question))

Question: What security measures protect against SIM swap fraud?
Retrieved 3 chunks:

Chunk 1 (page 8)
mitigate this with SS7 firewalls and anomaly detection systems, but the risk cannot be fully eliminated on legacy
protocols. 5G's use of HTTPS-based APIs (Service Based Architecture) substantially reduces this attack surface.
SIM Swap Fraud: Described in Section 5. Key mitigation: enforce strict in-person or multi-factor remote identity
verification before any SIM replacement. Flag accounts with recent SIM swaps for elevated fraud monitoring for
30 days.
International Revenue Share Fraud (IRSF): Fraudsters compromise a PBX or customer account and generate

Chunk 2 (page 5)
provide an additional layer of protection; after three incorrect PIN attempts the SIM is locked and requires a PUK
code to unlock.
SIM Swap Fraud: SIM swap attacks occur when a fraudster convinces a carrier to transfer a victim's number to a
new SIM. This allows the attacker to intercept SMS-based two-factor authent